In [ ]:
import os
import sys
import h5py
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# 路径设置。
THIS_DIR = os.getcwd()
MODULE_DIR = THIS_DIR
REPO_DIR = os.path.abspath(os.path.join(MODULE_DIR, '..'))

sys.path.insert(0, MODULE_DIR)

import tool  # noqa: E402
import importlib
from model import ParticleTransformerKD  # noqa: E402

importlib.reload(tool)  # noqa: E402

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# 实验输出目录。
RUN_NAME = 'smear_only_offline_hlt_hltkd_repeat3'
OUT_DIR = os.path.join(MODULE_DIR, 'runs', RUN_NAME)
FIG_DIR = os.path.join(OUT_DIR, 'figs')
TABLE_DIR = os.path.join(OUT_DIR, 'tables')
REPEAT_DIR = os.path.join(OUT_DIR, 'repeats')
CONFIG_PATH = os.path.join(OUT_DIR, 'config.json')

tool.ensure_dir(FIG_DIR)
tool.ensure_dir(TABLE_DIR)
tool.ensure_dir(REPEAT_DIR)

CONFIG = {
    'data_path': os.path.join(REPO_DIR, 'test.h5'),
    'n_jets': 200000,
    'max_particles': 100,
    'split_seed': 42,
    'repeat_seeds': [42, 52, 62],
    'eval_hlt_seed': 2026,
    'resmear_each_epoch': True,
    'hlt_effects': {
        'threshold_enabled': True,
        'smear_enabled': True,
        'merge_enabled': False,
        'merge_radius': 0.0,
        'efficiency_loss': 0.0,
        'pt_resolution': 0.10,
        'eta_resolution': 0.03,
        'phi_resolution': 0.03,
        'pt_threshold_offline': 0,
        'pt_threshold_hlt': 0,
    },
    'tagger': {
        'input_dim': 7,
        'embed_dim': 128,
        'num_heads': 8,
        'num_layers': 6,
        'ff_dim': 512,
        'dropout': 0.1,
    },
    'training': {
        'batch_size': 512,
        'epochs': 50,
        'lr': 5e-4,
        'weight_decay': 1e-5,
        'warmup_epochs': 3,
        'patience': 8,
        'feature_clip': 10.0,
        'min_delta': 1e-4,
    },
    'kd': {
        'temperature': 3.0,
        'alpha_kd': 0.5,
        'alpha_attn': 0,
    },
    'io': {
        'run_name': RUN_NAME,
        'out_dir': OUT_DIR,
        'fig_dir': FIG_DIR,
        'table_dir': TABLE_DIR,
        'repeat_dir': REPEAT_DIR,
        'config_path': CONFIG_PATH,
    },
}

tool.save_config(CONFIG, CONFIG_PATH)
print('Data path:', CONFIG['data_path'])
print('Run dir:', OUT_DIR)
print('Repeat seeds:', CONFIG['repeat_seeds'])
print('Per-epoch resmear:', CONFIG['resmear_each_epoch'])


In [2]:
# Load data
n = int(CONFIG['n_jets'])
S = int(CONFIG['max_particles'])

with h5py.File(CONFIG['data_path'], 'r') as f:
    labels = f['labels'][:n].astype(np.int64)
    weights = f['weights'][:n].astype(np.float32)
    pt = f['fjet_clus_pt'][:n, :S].astype(np.float32)
    eta = f['fjet_clus_eta'][:n, :S].astype(np.float32)
    phi = f['fjet_clus_phi'][:n, :S].astype(np.float32)
    E = f['fjet_clus_E'][:n, :S].astype(np.float32)

constituents_raw = np.stack([pt, eta, phi, E], axis=-1)  # [N,S,4]
masks_raw = pt > 0

print('Raw:', constituents_raw.shape, 'mask:', masks_raw.shape)
print('Signal:', int(labels.sum()), 'Bkg:', int((1 - labels).sum()))


Raw: (50000, 100, 4) mask: (50000, 100)
Signal: 25057 Bkg: 24943


In [ ]:
# 准备固定的 offline / eval-HLT 特征与数据划分。
prepared = tool.prepare_smear_baseline_data(
    constituents_raw,
    masks_raw,
    labels,
    weights,
    CONFIG,
    split_seed=int(CONFIG['split_seed']),
    eval_hlt_seed=int(CONFIG['eval_hlt_seed']),
    clip=float(CONFIG['training']['feature_clip']),
)

print('Offline features:', prepared.features_off_std.shape)
print('Eval HLT features:', prepared.features_hlt_eval_std.shape)
print(f"Split: train={len(prepared.train_idx):,} val={len(prepared.val_idx):,} test={len(prepared.test_idx):,}")


In [ ]:
# # 查看模型实际使用的标准化特征分布。
# feat_fig_path = os.path.join(FIG_DIR, 'feat_dists_train.png')

# print('Plotting feature distributions on TRAIN split (sampled)...')
# tool.plot_feat_dists(
#     prepared.features_off_std,
#     prepared.masks_off,
#     prepared.features_hlt_eval_std,
#     prepared.masks_hlt_eval,
#     jet_idx=prepared.train_idx,
#     title='Smear-only feature distributions (train split, sampled)',
#     bins=120,
#     max_vals=200_000,
#     clip=(-10, 10),
#     seed=int(CONFIG['eval_hlt_seed']),
#     save_path=feat_fig_path,
#     dpi=160,
# )


In [ ]:
# 三次重复训练并保存每次的 checkpoint / history / predictions。
repeat_results = []
for repeat_idx, repeat_seed in enumerate(CONFIG['repeat_seeds']):
    print()
    print(f"==== Repeat {repeat_idx + 1}/{len(CONFIG['repeat_seeds'])} seed={repeat_seed} ====")
    repeat_result = tool.run_smear_baseline_repeat(
        model_cls=ParticleTransformerKD,
        model_kwargs=CONFIG['tagger'],
        data=prepared,
        config=CONFIG,
        device=device,
        repeat_idx=repeat_idx,
        repeat_seed=int(repeat_seed),
        out_dir=REPEAT_DIR,
    )
    repeat_results.append(repeat_result)

print('Completed repeats:', len(repeat_results))


In [ ]:
# 汇总三次 repeat 的 AUC，并保存表格。
detail_rows, summary_rows = tool.build_repeat_metric_rows(repeat_results)

detail_path = os.path.join(TABLE_DIR, 'repeat_auc_detail.csv')
summary_path = os.path.join(TABLE_DIR, 'repeat_auc_summary.csv')
tool.save_rows_csv(detail_rows, detail_path)
tool.save_rows_csv(summary_rows, summary_path)

detail_df = pd.DataFrame(detail_rows).sort_values(['model_key', 'repeat_idx']).reset_index(drop=True)
summary_df = pd.DataFrame(summary_rows).sort_values('model_key').reset_index(drop=True)

summary_df_display = summary_df.copy()
if not summary_df_display.empty:
    summary_df_display['auc_mean'] = summary_df_display['auc_mean'].map(lambda x: f'{x:.6f}')
    summary_df_display['auc_std'] = summary_df_display['auc_std'].map(lambda x: f'{x:.6f}')

detail_df_display = detail_df[[
    'repeat_idx',
    'repeat_seed',
    'model_name',
    'test_auc',
    'best_val_auc',
    'prediction_path',
    'ckpt_path',
]].copy()
if not detail_df_display.empty:
    detail_df_display['test_auc'] = detail_df_display['test_auc'].map(lambda x: f'{x:.6f}')
    detail_df_display['best_val_auc'] = detail_df_display['best_val_auc'].map(lambda x: f'{x:.6f}')

print('Saved table:', detail_path)
print('Saved table:', summary_path)

try:
    from IPython.display import display
    display(summary_df_display)
    display(detail_df_display)
except Exception:
    print(summary_df_display.to_string(index=False))
    print(detail_df_display.to_string(index=False))


In [ ]:
# 画三次 repeat 的均值 ROC 和标准差带。
roc_fig_path = os.path.join(FIG_DIR, 'roc_test_mean_std.png')
tool.plot_repeat_mean_roc(repeat_results, roc_fig_path)
print('Prediction bundles are saved under:', REPEAT_DIR)


In [ ]:
# 从已保存的预测结果中临时提取固定 TPR 下的 FPR。
def fpr_at_target_tpr(labels: np.ndarray, preds: np.ndarray, target_tpr: float) -> float:
    fpr, tpr, _ = tool.compute_roc(np.asarray(labels), np.asarray(preds))
    tpr = np.asarray(tpr, dtype=np.float64)
    fpr = np.asarray(fpr, dtype=np.float64)
    unique_tpr, unique_idx = np.unique(tpr, return_index=True)
    unique_fpr = fpr[unique_idx]
    return float(np.interp(float(target_tpr), unique_tpr, unique_fpr))

example_path = detail_rows[0]['prediction_path']
bundle = np.load(example_path)
print('Example prediction bundle:', example_path)
print('Saved keys:', list(bundle.files))
print(f"Example FPR@TPR=0.30: {fpr_at_target_tpr(bundle['labels'], bundle['preds'], 0.30):.6e}")
print(f"Example FPR@TPR=0.50: {fpr_at_target_tpr(bundle['labels'], bundle['preds'], 0.50):.6e}")
